# Cleaning V3 Sample 1000 Validation

这个 Notebook 基于 `default_minio_dataset/sample_1000/raw.parquet` 验证清洗 v3 第一批算子，并展示运行产物、统计汇总和重点样本抽样。

## 1. Imports and setup

In [ ]:
from __future__ import annotations

import json

import pandas as pd

from image_gallery.cleaning import BasicCleaner
from notebooks._helpers.cleaning_configs import get_cleaning_v3_first_batch_operator_configs
from notebooks._helpers.datasets import (
    get_default_minio_sample_1000_raw_path,
    load_default_minio_sample_1000_dataset,
    load_default_minio_sample_1000_frame,
)
from notebooks._helpers.paths import get_notebook_library_root, reset_output_dir

NOTEBOOK_NAME = "cleaning_v3_sample_1000"
RUN_ROOT = reset_output_dir(get_notebook_library_root(NOTEBOOK_NAME))
RUN_OUTPUT_DIR = RUN_ROOT / "outputs"
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_path = get_default_minio_sample_1000_raw_path()
raw_frame = load_default_minio_sample_1000_frame()

RUN_ROOT, raw_path


## 2. Validate raw dataset

In [ ]:
assert "image_id" in raw_frame.columns
assert "image_uri" in raw_frame.columns
assert not raw_frame["image_uri"].isna().any()

print("raw_path:", raw_path)
print("raw_rows:", len(raw_frame))
print("raw_columns:", raw_frame.columns.tolist())
print("duplicated_image_id_count:", int(raw_frame["image_id"].duplicated().sum()))
raw_frame.head()


## 3. Load storage-backed dataset

In [ ]:
dataset = load_default_minio_sample_1000_dataset()

sample_uris = raw_frame["image_uri"].astype(str).head(3).tolist()
sample_images = []
for image_uri in sample_uris:
    image = dataset.read_image(image_uri)
    sample_images.append(
        {
            "image_uri": image_uri,
            "size": image.size,
            "format": image.format,
        }
    )

pd.DataFrame(sample_images)


## 4. Run BasicCleaner

In [ ]:
operator_configs = get_cleaning_v3_first_batch_operator_configs()
operator_names = [next(iter(item.keys())) for item in operator_configs]
print("operator_names:", operator_names)

cleaner = BasicCleaner(operator_configs)
cleaner.run(dataset, output_dir=RUN_OUTPUT_DIR)

preview = cleaner.preview()
state = cleaner.state()
context = cleaner._context
if context is None:
    raise AssertionError("cleaner context should exist")
paths = context.paths

parameter_table = pd.read_parquet(paths.parameter_table_path)
evaluation_table = pd.read_parquet(paths.evaluation_table_path)
parameter_manifest = json.loads(paths.parameter_manifest_path.read_text(encoding="utf-8"))
duplicate_pairs = pd.read_parquet(paths.relations_dir / "duplicate_pairs.parquet")

preview, state.head(), list(parameter_manifest.keys())[:10], duplicate_pairs.head()


## 5. Validate structured outputs

In [ ]:
required_parameter_columns = {
    "width",
    "height",
    "aspect_ratio",
    "megapixels",
    "blur_score",
    "brightness_score",
    "contrast_score",
    "blank_score",
    "content_hash",
    "exact_duplicate_group_id",
    "exact_duplicate_count",
}
required_evaluation_columns = {
    "decode_action",
    "decode_reason",
    "dimension_action",
    "dimension_reason",
    "aspect_ratio_action",
    "aspect_ratio_reason",
    "megapixel_action",
    "megapixel_reason",
    "blur_action",
    "blur_reason",
    "brightness_action",
    "brightness_reason",
    "contrast_action",
    "contrast_reason",
    "blank_action",
    "blank_reason",
    "exact_duplicate_action",
    "exact_duplicate_reason",
    "final_action",
    "final_reason",
    "triggered_operator_names",
}

assert required_parameter_columns.issubset(parameter_table.columns)
assert required_evaluation_columns.issubset(evaluation_table.columns)

print("parameter_table_shape:", parameter_table.shape)
print("evaluation_table_shape:", evaluation_table.shape)
print("state_columns:", state.columns.tolist())
print("parameter_manifest_count:", len(parameter_manifest))
print("duplicate_pairs_rows:", len(duplicate_pairs))


## 6. Summary statistics

In [ ]:
action_counts = (
    evaluation_table["final_action"]
    .value_counts(dropna=False)
    .rename_axis("final_action")
    .reset_index(name="count")
)

trigger_columns = [
    "decode_action",
    "dimension_action",
    "aspect_ratio_action",
    "megapixel_action",
    "blur_action",
    "brightness_action",
    "contrast_action",
    "blank_action",
    "exact_duplicate_action",
]
trigger_summary = pd.DataFrame(
    [
        {
            "column": column,
            "trigger_count": int((evaluation_table[column] != "keep").fillna(False).sum()),
        }
        for column in trigger_columns
    ]
)

duplicate_summary = pd.DataFrame(
    [
        {
            "duplicate_rows": int((parameter_table["exact_duplicate_count"] > 1).fillna(False).sum()),
            "duplicate_groups": int(parameter_table["exact_duplicate_group_id"].dropna().nunique()),
            "max_group_size": int(parameter_table["exact_duplicate_count"].fillna(0).max()),
        }
    ]
)

action_counts, trigger_summary, duplicate_summary


## 7. Sample interesting rows

In [ ]:
analysis_frame = parameter_table.merge(
    evaluation_table[
        [
            "image_id",
            "final_action",
            "final_reason",
            "blank_action",
            "blank_reason",
            "dimension_action",
            "dimension_reason",
            "exact_duplicate_action",
            "exact_duplicate_reason",
        ]
    ],
    on="image_id",
    how="left",
)


def sample_rows(frame: pd.DataFrame, query: str, limit: int = 5) -> pd.DataFrame:
    sampled = frame.query(query, engine="python").head(limit)
    columns = [
        "image_id",
        "image_uri",
        "blur_score",
        "brightness_score",
        "contrast_score",
        "blank_score",
        "exact_duplicate_count",
        "final_action",
        "final_reason",
        "blank_reason",
        "dimension_reason",
        "exact_duplicate_reason",
    ]
    existing_columns = [column for column in columns if column in sampled.columns]
    return sampled[existing_columns]


sample_rows(analysis_frame, "final_action == 'keep'"), \
sample_rows(analysis_frame, "final_action == 'review'"), \
sample_rows(analysis_frame, "final_action == 'drop'")


In [ ]:
sample_rows(analysis_frame, "blank_action != 'keep'"), \
sample_rows(analysis_frame, "dimension_action != 'keep'"), \
sample_rows(analysis_frame, "exact_duplicate_action != 'keep'")


## 8. Export validation

In [ ]:
full_export = cleaner.export("full", str(RUN_ROOT / "full.parquet")).to_frame()
dropped_export = cleaner.export("dropped", str(RUN_ROOT / "dropped.parquet")).to_frame()

assert len(full_export) == len(evaluation_table)
assert len(dropped_export) == int((evaluation_table["final_action"] == "drop").sum())

full_export.head(), dropped_export.head()
